# 논문의 분석 실험 직접 돌리기 (Transformer 쪽)

**목적.** 발표에서 논문 숫자를 그대로 가져다 쓴 실험을 직접 돌려 확인한다.

| 묶음 | 논문 | 내용 | 실행 수 | 예상 시간 (T4) |
|---|---|---|---|---|
| G0 | Table 8 | 효율: 파라미터 수, 추론 시간, 메모리 | 5 | 2분 |
| G1 | Table 4 | Informer → 단순화 4단계 (환율, ETTh1) | 32 | 30분~1시간 |
| G2 | Table 2 | FEDformer 등 Transformer 성능 (발표에 쓴 칸) | 17 | 2~4시간 |
| G3 | Table 6 | 위치·시간 임베딩 빼기 (교통) | 48 | 10~20시간 |
| G4 | Table 7 | 학습 데이터를 1년으로 줄이기 (교통) | 8 | 2~4시간 |
| G5 | Table 3 | 가까운 입력과 먼 입력 (전력, 교통, T=720) | 8 | 4~8시간 |
| G6 | Figure 4 | 입력 길이 바꾸기 (전력, 교통, T=720) | 154 | 수십 시간 |

시간은 추정이다. 실제 시간은 결과 파일의 `seconds` 열에 남는다.

**실행.** 런타임 **GPU → T4**(가능하면 더 큰 GPU) → **모두 실행**. 처음에 Google Drive 연결을 묻는다. 허용하면 결과가 Drive의 `dlinear_paper_exp/results.csv`에 쌓여서, 런타임이 끊겨도 다시 **모두 실행**하면 끝난 칸은 건너뛴다. 거절하면 `/content`에만 저장된다.

**순서.** 가벼운 묶음부터 돈다. 아래 `GROUPS`에서 돌릴 묶음을 고를 수 있다. 중간에 멈춰도 그때까지의 결과는 남는다.

**주의.** Table 3, Table 7, Table 4의 가운데 두 모델은 저자가 코드를 공개하지 않았다. 논문 본문의 설명대로 구현한 것이라, 숫자가 논문과 다르면 구현 해석의 차이일 수 있다. 자세한 내용은 2번 셀의 주석에 있다.

In [ ]:
# 0. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. 코드와 데이터 준비, 결과 저장 위치
import os, subprocess
os.chdir('/content')
subprocess.run('pip -q install einops gdown', shell=True)  # FEDformer needs einops
if not os.path.exists('LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git'], check=True)
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/data/traffic.csv'):
    subprocess.run('gdown -q --folder '
                   'https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy -O /content/raw_data && '
                   'find /content/raw_data -name "*.csv" -exec cp -n {} /content/data/ \\;', shell=True, check=True)
!git -C /content/LTSF-Linear log --oneline -1
!ls -la /content/data
OUTDIR = '/content'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/dlinear_paper_exp'
    os.makedirs(OUTDIR, exist_ok=True)
except Exception as e:
    print('Drive 연결 안 됨, /content에 저장:', e)
OUT = OUTDIR + '/results.csv'
print('결과 파일:', OUT)

In [ ]:
%%writefile /content/paper_exp.py
"""Runs one cell of the paper's analysis experiments on the official LTSF-Linear repository.

Usage:
  python paper_exp.py <repo_dir> <data_dir> key=value ...

Keys
  exp      t2 | emb | lookback | closefar | short | t4 | eff      (label only, except closefar/short/eff)
  model    FEDformer | Autoformer | Informer | Transformer | Linear | NLinear | DLinear
           | InfAttLin | InfEmbLin          (the two Table 4 variants, built here; see below)
  data     exchange | electricity | traffic | weather | ili | ETTh1 | ETTh2 | ETTm1 | ETTm2
  pred_len, seq_len (default 96; ili 36), embed_type (default 0), seed (default 2021)
  variant  closefar: close | far        short: ori | short
  epochs   overrides the official epoch count (official: exchange 1, traffic 3, weather 2, else 10)

Prints one line:
  RESULT,<exp>,<model>,<data>,<seq_len>,<pred_len>,<embed_type>,<variant>,<seed>,<mse>,<mae>,<mse_droplast>,<mae_droplast>,<epochs_run>,<seconds>
  (eff prints RESULT,eff,<model>,<params>,<ms_per_forward>,<peak_MiB>)

What is and is not official
  - Transformer / Informer / Autoformer / FEDformer and their settings come from the repository
    (scripts/EXP-LongForecasting/Formers_Long.sh, FEDformer/scripts/LongForecasting.sh).
  - embed_type 0..4 is the repository's own switch (scripts/EXP-Embedding).
  - closefar (Table 3), short (Table 7) and the two Table 4 variants have no public code.
    They are implemented here from the paper's text, so they are an interpretation:
      closefar: every sample is [far input 0..95][close input 96..191][target 192..192+T).
                'close' feeds 96..191, 'far' feeds 0..95. The decoder start tokens are the last
                label_len steps of whichever input is fed.
      short:    the training split is cut to its last 8,760 steps. The scaler is unchanged.
      InfAttLin: Informer with every self-attention (encoder and decoder) replaced by a
                 linear layer on the token features. Cross-attention is kept.
      InfEmbLin: Informer's input embedding, a linear map d_model -> channels, then a
                 linear map over time (seq_len -> pred_len). Nothing else.
"""
import os
import sys
import time

repo, data_dir = sys.argv[1], sys.argv[2]
kv = dict(a.split('=', 1) for a in sys.argv[3:])
EXP = kv.get('exp', 't2')
MODEL = kv['model']
DATA = kv.get('data', 'exchange')
P = int(kv.get('pred_len', 96))
SEED = int(kv.get('seed', 2021))
EMB = int(kv.get('embed_type', 0))
VAR = kv.get('variant', '-')
GAP = 96            # closefar: distance between the far input and the target
SHORT_N = int(kv.get('short_n', 8760))

CFG = {  # data_path, --data, channels, official epochs for the Formers
    'exchange': ('exchange_rate.csv', 'custom', 8, 1),
    'electricity': ('electricity.csv', 'custom', 321, 10),
    'traffic': ('traffic.csv', 'custom', 862, 3),
    'weather': ('weather.csv', 'custom', 21, 2),
    'ili': ('national_illness.csv', 'custom', 7, 10),
    'ETTh1': ('ETTh1.csv', 'ETTh1', 7, 10), 'ETTh2': ('ETTh2.csv', 'ETTh2', 7, 10),
    'ETTm1': ('ETTm1.csv', 'ETTm1', 7, 10), 'ETTm2': ('ETTm2.csv', 'ETTm2', 7, 10),
}
path, dname, C, off_ep = CFG[DATA]
L = int(kv.get('seq_len', 36 if DATA == 'ili' else 96))
LABEL = 18 if DATA == 'ili' else 48
linear = MODEL in ('Linear', 'NLinear', 'DLinear')
EPOCHS = int(kv.get('epochs', 10 if linear else off_ep))

fed = MODEL == 'FEDformer'
# embed_type labels used here: 0 model default, 1 all, 2 without positional, 3 without temporal, 4 without both.
# The FEDformer folder numbers them differently (0 without positional, 1 all, 2 without both, 3 without temporal).
EMB_ARG = {0: 0, 1: 1, 2: 0, 3: 3, 4: 2}[EMB] if fed else EMB
root = os.path.join(repo, 'FEDformer') if fed else repo
os.chdir(root)
sys.path.insert(0, root)

import numpy as np
import torch
import torch.nn as nn

if not hasattr(np, 'Inf'):  # NumPy 2 removed np.Inf, which the repository still uses
    np.Inf = np.inf

import data_provider.data_loader as dl
import exp.exp_main as em

# ---------------------------------------------------------------- data variants
if EXP == 'closefar':
    for cls in (dl.Dataset_Custom,):
        def _len(self):
            return len(self.data_x) - self.seq_len - GAP - self.pred_len + 1

        def _get(self, index):
            ib = index + (GAP if VAR == 'close' else 0)
            ie = ib + self.seq_len
            tb = index + GAP + self.seq_len
            te = tb + self.pred_len
            seq_x = self.data_x[ib:ie]
            seq_y = np.concatenate([self.data_x[ie - self.label_len:ie], self.data_y[tb:te]], 0)
            x_mark = self.data_stamp[ib:ie]
            y_mark = np.concatenate([self.data_stamp[ie - self.label_len:ie], self.data_stamp[tb:te]], 0)
            return seq_x, seq_y, x_mark, y_mark
        cls.__len__ = _len
        cls.__getitem__ = _get

if EXP == 'short' and VAR == 'short':
    _read = dl.Dataset_Custom.__read_data__

    def _read_short(self):
        _read(self)
        if self.set_type == 0:
            self.data_x = self.data_x[-SHORT_N:]
            self.data_y = self.data_y[-SHORT_N:]
            self.data_stamp = self.data_stamp[-SHORT_N:]
    dl.Dataset_Custom.__read_data__ = _read_short

# ---------------------------------------------------------------- Table 4 variants
if MODEL in ('InfAttLin', 'InfEmbLin'):
    from models import Informer
    from layers.Embed import DataEmbedding

    class _LinAttn(nn.Module):
        def __init__(self, d):
            super().__init__()
            self.lin = nn.Linear(d, d)

        def forward(self, q, k, v, attn_mask=None):
            return self.lin(q), None

    class _AttLin(nn.Module):
        def __init__(self, a):
            super().__init__()
            self.m = Informer.Model(a)
            for layer in self.m.encoder.attn_layers:
                layer.attention = _LinAttn(a.d_model)
            for layer in self.m.decoder.layers:
                layer.self_attention = _LinAttn(a.d_model)

        def forward(self, *x, **k):
            return self.m(*x, **k)

    class _EmbLin(nn.Module):
        def __init__(self, a):
            super().__init__()
            self.emb = DataEmbedding(a.enc_in, a.d_model, a.embed, a.freq, a.dropout)
            self.proj = nn.Linear(a.d_model, a.c_out)
            self.time = nn.Linear(a.seq_len, a.pred_len)

        def forward(self, x_enc, x_mark_enc, *a, **k):
            o = self.proj(self.emb(x_enc, x_mark_enc))
            return self.time(o.permute(0, 2, 1)).permute(0, 2, 1)

    _build = em.Exp_Main._build_model

    def _build_custom(self):
        if self.args.model == 'InfAttLin':
            return _AttLin(self.args).float()
        if self.args.model == 'InfEmbLin':
            return _EmbLin(self.args).float()
        return _build(self)
    em.Exp_Main._build_model = _build_custom

# ---------------------------------------------------------------- scoring
T0 = time.time()
STATE = {'epochs': 0}
_vali = em.Exp_Main.vali
_calls = {'n': 0}


def _vali_count(self, *a, **k):  # vali is called twice per epoch (val, test) in the repository
    _calls['n'] += 1
    return _vali(self, *a, **k)


em.Exp_Main.vali = _vali_count


def _forward(self, bx, by, bxm, bym):
    a = self.args
    if 'Linear' in a.model:
        return self.model(bx)
    dec = torch.cat([by[:, :a.label_len, :], torch.zeros_like(by[:, -a.pred_len:, :])], dim=1).float()
    out = self.model(bx, bxm, dec, bym)
    return out[0] if isinstance(out, (tuple, list)) else out


def test(self, setting, test=0):
    _, loader = self._get_data(flag='test')
    self.model.eval()
    a = self.args
    se, ae = [], []
    with torch.no_grad():
        for bx, by, bxm, bym in loader:
            bx = bx.float().to(self.device); by = by.float().to(self.device)
            bxm = bxm.float().to(self.device); bym = bym.float().to(self.device)
            out = _forward(self, bx, by, bxm, bym)
            d = (out[:, -a.pred_len:, :] - by[:, -a.pred_len:, :]).double()
            se.append((d ** 2).mean(dim=(1, 2)).cpu()); ae.append(d.abs().mean(dim=(1, 2)).cpu())
    se = torch.cat(se).numpy(); ae = torch.cat(ae).numpy()
    m = (len(se) // a.batch_size) * a.batch_size  # what a drop_last=True test loader would score
    print(f'RESULT,{EXP},{MODEL},{DATA},{L},{P},{EMB},{VAR},{SEED},{se.mean():.6f},{ae.mean():.6f},'
          f'{se[:m].mean():.6f},{ae[:m].mean():.6f},{_calls["n"] // 2},{time.time() - T0:.0f}', flush=True)


em.Exp_Main.test = test

if EXP == 'eff':
    def measure(self, setting):
        a = self.args
        dev = self.device
        self.model.eval()
        n = sum(p.numel() for p in self.model.parameters())
        bx = torch.randn(1, a.seq_len, a.enc_in, device=dev); bxm = torch.randn(1, a.seq_len, 4, device=dev)
        by = torch.randn(1, a.label_len + a.pred_len, a.enc_in, device=dev)
        bym = torch.randn(1, a.label_len + a.pred_len, 4, device=dev)
        cuda = dev.type == 'cuda'
        with torch.no_grad():
            for _ in range(3):
                _forward(self, bx, by, bxm, bym)
            if cuda:
                torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
            ts = []
            for _ in range(5):
                t = time.time(); _forward(self, bx, by, bxm, bym)
                if cuda:
                    torch.cuda.synchronize()
                ts.append(time.time() - t)
        peak = torch.cuda.max_memory_allocated() / 2 ** 20 if cuda else float('nan')
        print(f'RESULT,eff,{MODEL},{n},{1000 * sum(ts) / 5:.2f},{peak:.0f}', flush=True)
        return self.model
    em.Exp_Main.train = measure
    em.Exp_Main.test = lambda self, setting, test=0: None

# ---------------------------------------------------------------- run the repository's own entry script
tag = f'px_{EXP}_{MODEL}_{DATA}_{L}_{P}_{EMB}_{VAR}_{SEED}'
argv = ['--is_training', '1', '--root_path', data_dir, '--data_path', path, '--data', dname,
        '--features', 'M', '--seq_len', str(L), '--label_len', str(LABEL), '--pred_len', str(P),
        '--enc_in', str(C), '--dec_in', str(C), '--c_out', str(C), '--des', 'Exp', '--itr', '1',
        '--num_workers', '0', '--train_epochs', str(EPOCHS), '--model', MODEL,
        '--e_layers', '2', '--d_layers', '1', '--factor', '3', '--embed_type', str(EMB_ARG)]
if linear:
    argv += ['--learning_rate', kv.get('lr', '0.005'), '--batch_size', kv.get('bs', '32')]
argv += ['--task_id', tag] if fed else ['--model_id', tag]
sys.argv = ['run.py' if fed else 'run_longExp.py'] + argv
src = open(sys.argv[0]).read().replace('fix_seed = 2021', f'fix_seed = {SEED}')
exec(compile(src, sys.argv[0], 'exec'), {'__name__': '__main__'})

## 2-가. 남은 두 칸: 혼합 정밀도(AMP)로 돌리면 달라지는가
원인을 못 찾은 두 칸을 `--use_amp` 옵션으로 돌린다. 저자가 이 옵션을 켰다면 계산 정밀도가 달라져 마지막 자리가 바뀔 수 있다. 10분 안팎.

| 칸 | 코드 | 기본 결과 | 논문 |
|---|---|---|---|
| 다변량 DLinear ETTm2 96 | 2022-06 (`04ffd8d`) | 0.16766 / 0.25980 | 0.167 / 0.260 |
| NLinear 환율 720 | 2022-08 (`7f7a25b`) | 0.94301 / 0.72828 | 1.033 / 0.780 |

In [ ]:
import re, time
AMP = OUTDIR + '/amp.csv'
if not os.path.exists(AMP):
    os.chdir('/content/LTSF-Linear')
    for name, commit in [('june', '04ffd8d'), ('aug', '7f7a25b')]:
        d = f'/content/{name}'
        if not os.path.exists(d):
            r = subprocess.run(['git', 'worktree', 'add', '-q', d, commit])
            if r.returncode != 0:
                subprocess.run(['git', 'fetch', '-q', '--unshallow'])
                subprocess.run(['git', 'worktree', 'add', '-q', d, commit], check=True)
        subprocess.run(f"sed -i 's/np\\.Inf/np.inf/' {d}/utils/tools.py", shell=True, check=True)
        os.makedirs(f'{d}/dataset', exist_ok=True)
        subprocess.run(f'cp -n /content/data/ETTm2.csv /content/data/exchange_rate.csv {d}/dataset/', shell=True)
    RUNS = [
      ('DLinear ETTm2 96', 'june', '--data_path ETTm2.csv --data ETTm2 --features M --seq_len 336 --pred_len 96 --enc_in 7 --model DLinear --batch_size 32 --learning_rate 0.001'),
      ('NLinear Exchange 720', 'aug', '--data_path exchange_rate.csv --data custom --features M --seq_len 336 --pred_len 720 --enc_in 8 --model NLinear --batch_size 32 --learning_rate 0.0005'),
    ]
    rows = ['cell,amp,rep,mse,mae']
    for cell, d, args in RUNS:
        for amp in ['', '--use_amp']:
            for rep in range(2):
                cmd = f'cd /content/{d} && python -u run_longExp.py --is_training 1 --root_path ./dataset/ --model_id amp{rep} --des Exp --itr 1 {args} {amp}'
                r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
                m = re.search(r'^mse:([0-9.]+), mae:([0-9.]+)', r.stdout, re.M)
                rows.append(f"{cell},{'amp' if amp else 'fp32'},{rep+1},{m.group(1) if m else 'FAIL'},{m.group(2) if m else (r.stdout + r.stderr)[-200:].replace(chr(10), ' ')}")
                print(rows[-1]); subprocess.run(f'rm -rf /content/{d}/checkpoints', shell=True)
    open(AMP, 'w').write('\n'.join(rows) + '\n')
    os.chdir('/content')
print(open(AMP).read())

## 3. 실행
돌릴 묶음을 `GROUPS`에 적는다. 기본은 전부, 가벼운 것부터다.

In [ ]:
GROUPS = ['G0', 'G1', 'G2', 'G3', 'G4', 'G5', 'G6']

import time, pandas as pd
Q = []
def add(g, **k): Q.append((g, k))
H = [96, 192, 336, 720]
# G0 Table 8
for m in ['DLinear', 'Transformer', 'Informer', 'Autoformer', 'FEDformer']:
    add('G0', exp='eff', model=m, data='electricity', pred_len=720)
# G1 Table 4
for d, lr in [('exchange', '0.0005'), ('ETTh1', '0.005')]:
    for p in H:
        add('G1', exp='t4', model='Linear', data=d, pred_len=p, lr=lr)
        if d == 'exchange' and p in (96, 192): add('G1', exp='t4', model='Linear', data=d, pred_len=p, lr=lr, bs=8, variant='bs8')
        for m in ['InfEmbLin', 'InfAttLin', 'Informer']:
            add('G1', exp='t4', model=m, data=d, pred_len=p)
# G2 Table 2 (the cells used in the talk)
for m in ['FEDformer', 'Autoformer', 'Informer']:
    for p in H:
        add('G2', exp='t2', model=m, data='exchange', pred_len=p)
for d, p in [('ETTh2', 96), ('ili', 24), ('weather', 96), ('electricity', 96), ('traffic', 96)]:
    add('G2', exp='t2', model='FEDformer', data=d, pred_len=p)
# G3 Table 6: 1 all, 2 without positional, 3 without temporal, 4 without both
for m in ['FEDformer', 'Autoformer', 'Informer']:
    for e in [1, 2, 3, 4]:
        for p in H:
            add('G3', exp='emb', model=m, data='traffic', pred_len=p, embed_type=e)
# G4 Table 7 (the full-data column is G3 with embed_type 2)
for m in ['FEDformer', 'Autoformer']:
    for p in H:
        add('G4', exp='short', model=m, data='traffic', pred_len=p, variant='short', embed_type=2)
# G5 Table 3
for d in ['electricity', 'traffic']:
    for m in ['FEDformer', 'Autoformer']:
        for v in ['close', 'far']:
            add('G5', exp='closefar', model=m, data=d, pred_len=720, variant=v)
# G6 Figure 4
LB = [48, 72, 96, 120, 144, 168, 192, 336, 504, 672, 720]
for d, lr in [('electricity', '0.001'), ('traffic', '0.05')]:
    for m in ['DLinear', 'NLinear', 'Linear']:
        for L in LB:
            add('G6', exp='lookback', model=m, data=d, pred_len=720, seq_len=L, lr=lr, bs=16)
    for m in ['FEDformer', 'Autoformer', 'Informer', 'Transformer']:
        for L in LB:
            add('G6', exp='lookback', model=m, data=d, pred_len=720, seq_len=L)

# one-time fix (2026-11-21): FEDformer numbers embed_type differently, so its earlier 'embed_type 2' rows were really 'without both'
FIX = OUTDIR + '/fix_fed_embed.done'
if os.path.exists(OUT) and not os.path.exists(FIX):
    d = pd.read_csv(OUT)
    m_ = (d.model == 'FEDformer') & (d.exp == 'emb') & (d.embed_type == 2)
    d.loc[m_, 'embed_type'] = 4
    d = d[~((d.model == 'FEDformer') & (d.exp == 'short'))]
    d.to_csv(OUT, index=False); print('FEDformer 행', int(m_.sum()), '개의 표시를 고쳤습니다')
open(FIX, 'w').write('done')
COLS = 'group,exp,model,data,seq_len,pred_len,embed_type,variant,seed,mse,mae,mse_droplast,mae_droplast,epochs_run,seconds'
EFF = OUTDIR + '/efficiency.csv'
if not os.path.exists(OUT): open(OUT, 'w').write(COLS + '\n')
if not os.path.exists(EFF): open(EFF, 'w').write('model,params,ms_per_forward,peak_MiB\n')
def key(k):
    L = k.get('seq_len', 36 if k['data'] == 'ili' else 96)
    return (k['exp'], k['model'], k['data'], int(L), int(k['pred_len']), int(k.get('embed_type', 0)), k.get('variant', '-'))
os.makedirs(OUTDIR + '/logs', exist_ok=True)
todo = [(g, k) for g, k in Q if g in GROUPS]
print(len(todo), '개 실행 예정')
for i, (g, k) in enumerate(todo):
    d = pd.read_csv(OUT); done = set(zip(d.exp, d.model, d.data, d.seq_len, d.pred_len, d.embed_type, d.variant))
    if k['exp'] == 'eff':
        if k['model'] in set(pd.read_csv(EFF).model): continue
    elif key(k) in done:
        continue
    t = time.time()
    r = subprocess.run(['python', '-u', '/content/paper_exp.py', '/content/LTSF-Linear', '/content/data/'] + [f'{a}={b}' for a, b in k.items()],
                       capture_output=True, text=True)
    name = '_'.join(str(v) for v in k.values())
    open(f'{OUTDIR}/logs/{name}.log', 'w').write(r.stdout[-20000:] + r.stderr[-5000:])
    rows = [l[7:] for l in r.stdout.splitlines() if l.startswith('RESULT,')]
    if not rows:
        print(f'[{i+1}/{len(todo)}] {g} {name}: 실패 ({time.time()-t:.0f}초)'); print((r.stdout + r.stderr)[-800:]); continue
    if k['exp'] == 'eff':
        open(EFF, 'a').write(rows[0][4:] + '\n')
    else:
        open(OUT, 'a').write(g + ',' + rows[0] + '\n')
    print(f'[{i+1}/{len(todo)}] {g} {name}: {rows[0]} ({time.time()-t:.0f}초)', flush=True)
    subprocess.run('rm -rf /content/LTSF-Linear/checkpoints /content/LTSF-Linear/FEDformer/checkpoints', shell=True)

## 4. 결과를 채팅에 붙여 주세요
아래 셀 출력을 그대로 복사해 주시면 됩니다. 일부만 끝났어도 괜찮습니다.

In [ ]:
print(open(AMP).read())
print(open(EFF).read())
print(open(OUT).read())